# Customer Data Cleaning
This notebook handles data cleaning, missing value imputation, outlier removal, data type corrections, zero-variance column removal, categorical standardization, and feature engineering for the retail customer dataset.

## 0. Load Dataset
We load the raw customer dataset from the Excel file.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_excel("../dataset/raw/data_market.xlsx")
print(f"Initial shape: {df.shape}")

## 1. Missing Value Treatment
First, we analyze missing values. The only column with missing values is `Income` (24 missing values).

### Imputation Strategy:
We will impute the missing `Income` values using the **median income of the customer's respective Education level**.

**Why this strategy?**
Income is highly correlated with educational attainment. Imputing with the overall median or mean would ignore this structural relationship, whereas grouping by `Education` preserves the income distribution characteristics of different educational segments (e.g., PhD vs. Basic).

In [ ]:
# Create a copy of the dataframe to store cleaned data
cleaned_df = df.copy()

# Show missing values before treatment
print("Missing values before imputation:")
print(cleaned_df.isnull().sum()[cleaned_df.isnull().sum() > 0])

# Impute Income using median of Education group
cleaned_df['Income'] = cleaned_df.groupby('Education')['Income'].transform(lambda x: x.fillna(x.median()))

# Verify missing values after treatment
print("\nMissing values after imputation:")
print(cleaned_df.isnull().sum()[cleaned_df.isnull().sum() > 0])

## 2. Duplicate Handling
We verify if there are any duplicate records in the dataset and remove them if present.

In [ ]:
# Check for duplicates
duplicates = cleaned_df.duplicated().sum()
print(f"Number of duplicate records found: {duplicates}")

# Remove duplicates if any exist
if duplicates > 0:
    cleaned_df = cleaned_df.drop_duplicates()
    print("Duplicates removed.")
else:
    print("No duplicates to remove.")

## 3. Data Type Corrections
The customer registration date column `Dt_Customer` is currently stored as a string or object. We convert it to a proper `datetime64` data type.

In [ ]:
# Convert Dt_Customer to datetime
cleaned_df['Dt_Customer'] = pd.to_datetime(cleaned_df['Dt_Customer'])
print(f"Dt_Customer data type: {cleaned_df['Dt_Customer'].dtype}")

## 4. Zero-Variance Columns
We identify and remove columns that have only a single unique value (zero variance), as they do not provide any predictive power or information for analytics.

In [ ]:
# Detect constant columns
constant_cols = [col for col in cleaned_df.columns if cleaned_df[col].nunique() <= 1]
print(f"Constant columns detected: {constant_cols}")

# Remove constant columns
cleaned_df = cleaned_df.drop(columns=constant_cols)
print(f"Remaining columns: {cleaned_df.shape[1]}")

## 5. Categorical Cleaning
We standardize the values in the `Marital_Status` column to clean up anomalous/non-standard categories.

### Replacements:
* `Alone` -> `Single` (conceptually identical)
* `Absurd` -> `Single` (treated as single/unspecified)
* `YOLO` -> `Single` (treated as single/unspecified)

All other categories (`Single`, `Together`, `Married`, `Divorced`, `Widow`) are standard and will be kept as-is.

In [ ]:
# Display unique values before cleaning
print("Unique Marital_Status values before cleaning:")
print(cleaned_df['Marital_Status'].unique())

# Define standardizing mapping
marital_mapping = {
    'Alone': 'Single',
    'Absurd': 'Single',
    'YOLO': 'Single'
}

# Apply mapping (leaving unmapped values as-is)
cleaned_df['Marital_Status'] = cleaned_df['Marital_Status'].replace(marital_mapping)

# Display unique values after cleaning
print("\nUnique Marital_Status values after cleaning:")
print(cleaned_df['Marital_Status'].unique())

## 6. Outlier Analysis
We inspect and handle extreme outliers in `Year_Birth` and `Income`.

### Findings & Decision:
1. **Year_Birth**: There are 3 records with birth years before 1940 (specifically 1893, 1899, and 1900). Since this represents ages above 120 (clearly data entry errors or placeholder values), we will **remove** these 3 records.
2. **Income**: There is an extreme outlier of $666,666 (Index 2233). This customer has a very high income but extremely low spending ($62 total), indicating a data entry error. We will **remove** this single record.

We keep other high-income records (~$150k - $162k) because they show corresponding high-spending patterns (e.g. premium luxury customers) and are plausible.

In [ ]:
# Initial shape
initial_rows = cleaned_df.shape[0]

# Detect and remove Year_Birth outliers (birth year < 1940)
birth_outliers = cleaned_df[cleaned_df['Year_Birth'] < 1940]
print(f"Removing {len(birth_outliers)} birth year outliers: {birth_outliers['Year_Birth'].tolist()}")
cleaned_df = cleaned_df[cleaned_df['Year_Birth'] >= 1940]

# Detect and remove extreme Income outliers (Income > 600000)
income_outliers = cleaned_df[cleaned_df['Income'] > 600000]
print(f"Removing {len(income_outliers)} extreme income outliers: {income_outliers['Income'].tolist()}")
cleaned_df = cleaned_df[cleaned_df['Income'] <= 600000]

print(f"\nTotal rows removed: {initial_rows - cleaned_df.shape[0]}")

## 7. Feature Engineering
We construct two new features from the existing temporal columns:
1. **Age**: Calculated as `2014 - Year_Birth` (since 2014 is the collection year of this dataset, matching the latest customer registration).
2. **Customer_Tenure**: Calculated as the number of days between the customer's registration date (`Dt_Customer`) and the maximum sign-up date in the dataset (representing the end of data collection).

In [ ]:
# Create Age (using 2014 as reference year for data collection)
cleaned_df['Age'] = 2014 - cleaned_df['Year_Birth']

# Create Customer_Tenure (days since sign-up relative to the max registration date)
reference_date = cleaned_df['Dt_Customer'].max()
cleaned_df['Customer_Tenure'] = (reference_date - cleaned_df['Dt_Customer']).dt.days

# Show engineered features
cleaned_df[['Year_Birth', 'Age', 'Dt_Customer', 'Customer_Tenure']].head()

## 8. Validation
We validate the final structure, types, missing values, and statistics of our cleaned dataset.

In [ ]:
print(f"Final shape of the cleaned dataset: {cleaned_df.shape}\n")
print("--- Dataframe Info ---")
cleaned_df.info()

print("\n--- Missing Values Check ---")
print(cleaned_df.isnull().sum())

print("\n--- Statistical Summary ---")
cleaned_df.describe(include='all')

## 9. Export Cleaned Dataset
Finally, we export the cleaned dataset as a CSV file to the `dataset/processed` folder.

In [ ]:
# Create directory if it does not exist
os.makedirs("../dataset/processed", exist_ok=True)

# Export cleaned data
export_path = "../dataset/processed/cleaned_customer_data.csv"
cleaned_df.to_csv(export_path, index=False)
print(f"Cleaned dataset successfully exported to: {export_path}")